# 04. Grouped FIFO PWM для многоканальной схемы

Этот ноутбук показывает вариант, где FIFO read throughput и число физических PWM-каналов задаются отдельно:

```text
samples_per_period = сколько FIFO-сэмплов читается за один PWM-период
channels           = сколько физических PWM-каналов суммируется
```

Функция модели: `pwm_kind2_fifo_grouped_multichannel(...)`.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if (HERE / "tutorial_helpers.py").exists():
    TUTORIAL_DIR = HERE
elif (HERE / "tutorials" / "tutorial_helpers.py").exists():
    TUTORIAL_DIR = HERE / "tutorials"
else:
    raise RuntimeError("Run this notebook from the repository root or from the tutorials folder")

path_text = str(TUTORIAL_DIR)
if path_text not in sys.path:
    sys.path.insert(0, path_text)

import matplotlib.pyplot as plt
import numpy as np

from tutorial_helpers import (
    configure_plots,
    grouped_fifo_channel_waveforms,
    load_pwm_lab,
    plot_bitstream,
    plot_channel_stack,
    plot_moving_average_reconstruction,
    plot_pwm_carrier_output,
    plot_spectra,
    print_peak_table,
    pwm_kind2_channel_waveforms,
    show_grouped_mapping,
    time_us,
)

pl = load_pwm_lab()
configure_plots()

## Параметры схемы

Пример ниже читает два FIFO-сэмпла за PWM-период и распределяет их по четырем физическим каналам. Каждый FIFO-сэмпл в группе получает две физические копии.

In [ ]:
config = pl.PwmConfig(f_clk=80e6, f_pwm=1e6, resolution_bits=8)
samples_per_period = 2
channels = 4
f_data = samples_per_period * config.actual_f_pwm
f_signal = 60e3
n_groups = 128
n_samples = n_groups * samples_per_period

_, x_fifo = pl.sine_samples(freq=f_signal, sample_rate=f_data, n_samples=n_samples, amplitude=0.85)

print(f"f_pwm              = {config.actual_f_pwm / 1e6:.3f} MHz")
print(f"f_data             = {f_data / 1e6:.3f} MHz")
print(f"samples_per_period = {samples_per_period}")
print(f"channels           = {channels}")

## Mapping FIFO-сэмплов на каналы

In [ ]:
show_grouped_mapping(samples_per_period=samples_per_period, channels=channels, groups=6);

## Временная реализация каналов и суммы

In [ ]:
channel_traces, grouped = grouped_fifo_channel_waveforms(
    x_fifo,
    config,
    samples_per_period=samples_per_period,
    channels=channels,
)
raw_sum = channel_traces.sum(axis=0)
normalized_sum = raw_sum / channels

plot_channel_stack(
    channel_traces,
    sample_rate=config.f_clk,
    max_points=8 * config.period_samples,
    summed=normalized_sum,
    title="Grouped FIFO PWM channels",
);

## Проверка среднего за период

Ожидаемая низкочастотная величина здесь — среднее группы FIFO-сэмплов, а не каждый исходный FIFO-сэмпл по отдельности.

In [ ]:
y_grouped = pl.pwm_kind2_fifo_grouped_multichannel(
    x_fifo,
    config,
    samples_per_period=samples_per_period,
    channels=channels,
    normalize_sum=True,
)

group_average = grouped.mean(axis=1)
period_average = pl.moving_average_decimate(y_grouped, config.period_samples)
t_ms = np.arange(group_average.size) / config.actual_f_pwm * 1e3

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
ax.plot(t_ms, group_average, "o-", markersize=3, label="FIFO group average")
ax.plot(t_ms, period_average, "s--", markersize=3, label="PWM period average")
ax.set_title("Grouped FIFO low-frequency envelope")
ax.set_xlabel("time, ms")
ax.set_ylabel("normalized amplitude")
ax.legend(loc="upper right");

## Same-phase против phase-interleaved для grouped FIFO

In [ ]:
y_same_phase = pl.pwm_kind2_fifo_grouped_multichannel(
    x_fifo,
    config,
    samples_per_period=samples_per_period,
    channels=channels,
    phase_offsets=np.zeros(channels),
    normalize_sum=True,
)

y_interleaved = pl.pwm_kind2_fifo_grouped_multichannel(
    x_fifo,
    config,
    samples_per_period=samples_per_period,
    channels=channels,
    normalize_sum=True,
)

plot_spectra(
    {
        "group average": np.repeat(group_average, config.period_samples),
        "same phase": y_same_phase,
        "interleaved": y_interleaved,
    },
    sample_rate=config.f_clk,
    f_max=10e6,
    f_scale=1e6,
    f_unit="MHz",
    title="Grouped FIFO PWM spectrum",
);

## Низкочастотный спектр огибающей

Здесь анализируем уже не сам PWM-поток на `f_clk`, а последовательность средних значений с частотой `f_pwm`.

In [ ]:
plot_spectra(
    {
        "FIFO group average": group_average,
        "PWM period average": period_average,
    },
    sample_rate=config.actual_f_pwm,
    f_max=250e3,
    f_scale=1e3,
    f_unit="kHz",
    title="Low-frequency envelope spectrum",
);

print_peak_table(
    {"group average": group_average, "period average": period_average},
    sample_rate=config.actual_f_pwm,
    f_min=1.0,
    f_max=250e3,
    f_scale=1e3,
    f_unit="kHz",
)